# Deneme 01 · Veri ve çeviri

Atılabilir deneme: kalıcı kod Hafta 2'de `src/ruya/` altına yazılır. Sonuçlar `docs/deneyler/01-veri-ve-ceviri.md` dosyasına işlenir; H1–H4 hipotezleri ve eşikleri orada tanımlıdır.

`# ELLE DOLDUR` işaretli hücreler elle doldurulur; diğer hücreler sırayla çalıştırılır.

In [ ]:
# Kurulum hücresi: bütün notebook'larda aynı (bkz. notebooks/README.md)
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
%run {REPO}/notebooks/_kurulum.py

## 0. Kaynakları indir

Nablusi ve Miller `data/raw` klasörüne iner; daha önce indirilmiş dosyalar atlanır.

In [ ]:
import subprocess
import sys

from ruya.config import ayarlar

subprocess.run(
    [
        sys.executable,
        str(REPO / "scripts" / "download_sources.py"),
        "--kaynak",
        "nablusi",
        "miller",
    ],
    check=True,
)

ozet_yolu = ayarlar().ham_klasoru / "KAYNAK_OZETLERI.json"
print(ozet_yolu.read_text(encoding="utf-8"))


## 1. Ayrıştırma işlevleri

Aşağıdaki hücre yalnızca tanım içerir (veri okumaz), bu yüzden veri olmadan da denetlenebilir.

In [ ]:
import re

SAYFA_DESENI = re.compile(r"PageV(\d+)P(\d+)")
SAYFA_SATIRI = re.compile(r"PageV\d+P\d+")
MS_DESENI = re.compile(r"\bms\d+\b")
BASLIK_DESENI = re.compile(r"^\(([^)]*)\)")
BAB_DESENI = re.compile(r"^###\s*\|+\s*")
ALT_ISARETLERI = ("رأى", "رآ", "رؤية")


def nablusi_paragraflari(satirlar: list[str]) -> list[dict]:
    """OpenITI mARkdown metnini paragraflara böler (bkz. docs/02)."""
    basla = 0
    for i, satir in enumerate(satirlar):
        if "#META#Header#End#" in satir:
            basla = i + 1
            break

    paragraflar: list[dict] = []
    bab = None
    acik: list[str] | None = None
    for i in range(basla, len(satirlar)):
        satir = satirlar[i]
        if satir.startswith("### |"):
            bab = BAB_DESENI.sub("", satir).strip()
            acik = None
            continue
        if SAYFA_SATIRI.fullmatch(satir.strip().lstrip("#").strip()):
            continue  # yalnızca sayfa işareti: paragraf başlatmaz
        if satir.startswith("# "):
            acik = [satir]
            paragraflar.append({"bab": bab, "satir": i + 1, "ham": acik})
            continue
        if satir.startswith("~~") and acik is not None:
            acik.append(satir)

    # OpenITI'de işaret sayfayı kapatır: paragraf, kendisinden sonra görülen ilk
    # işaretin sayfasını alır (satır içi işaret de sayılır).
    isaretler = []
    for i in range(basla, len(satirlar)):
        eslesme = SAYFA_DESENI.search(satirlar[i])
        if eslesme:
            isaretler.append((i, f"V{eslesme.group(1)}P{eslesme.group(2)}"))

    sonuc = []
    for no, p in enumerate(paragraflar):
        parcalar = []
        for satir in p["ham"]:
            s = satir[1:] if satir.startswith("#") else satir
            s = s.strip()
            if s.startswith("- "):
                s = s[2:]
            parcalar.append(s)
        metin = " ".join(parcalar)
        metin = metin.replace("~~", " ")
        metin = SAYFA_DESENI.sub(" ", metin)
        metin = MS_DESENI.sub(" ", metin)
        metin = " ".join(metin.split())

        baslik_eslesmesi = BASLIK_DESENI.match(metin)
        baslik = baslik_eslesmesi.group(1) if baslik_eslesmesi else None
        if baslik is None:
            tur = "diger"
        elif any(isaret in baslik for isaret in ALT_ISARETLERI):
            tur = "alt"
        else:
            tur = "baslikli"

        sayfa = next((s for i, s in isaretler if i >= p["satir"] - 1), None)
        sonuc.append(
            {
                "no": no,
                "bab": p["bab"],
                "tur": tur,
                "baslik": baslik,
                "sayfa": sayfa,
                "satir": p["satir"],
                "satir_sayisi": len(p["ham"]),
                "metin": metin,
            }
        )
    return sonuc


def miller_maddeleri(metin: str) -> list[dict]:
    """Gutenberg #926 metnini başlıklı maddelere böler."""
    satirlar = metin.splitlines()
    basla = 0
    for i, satir in enumerate(satirlar):
        if satir.startswith("*** START OF"):
            basla = i + 1
            break
    son = len(satirlar)
    for i in range(basla, len(satirlar)):
        if satirlar[i].startswith("*** END OF"):
            son = i
            break

    maddeler: list[dict] = []
    baslik: str | None = None
    satir_no = 0
    paragraflar: list[str] = []
    tampon: list[str] = []

    def _paragraf_kapat() -> None:
        if tampon:
            paragraflar.append(" ".join(tampon))
            tampon.clear()

    def _madde_kapat() -> None:
        _paragraf_kapat()
        if baslik is not None:
            govde = "\n\n".join(paragraflar)
            govde = re.sub(r"\[\d+\]", "", govde)
            govde = re.sub(r"[ \t]{2,}", " ", govde).strip()
            maddeler.append({"baslik": baslik, "metin": govde, "satir": satir_no})

    for i in range(basla, son):
        satir = satirlar[i]
        eslesme = re.match(r"^_([^_]+)_\.(\[\d+\])?$", satir.strip())
        if eslesme:
            _madde_kapat()
            baslik = eslesme.group(1)
            satir_no = i + 1
            paragraflar = []
            tampon = []
            continue
        if baslik is None:
            continue  # ilk başlıktan önceki satırlar atılır
        if not satir.strip():
            _paragraf_kapat()
        else:
            tampon.append(satir.strip())
    _madde_kapat()
    return maddeler

## 2. H1 · Nablusi: "ح" babından 50 paragraf

Beklenen (docs/02): ~5746 paragraf, ~2400 başlıklı, ~3300 alt, 29 bab.

In [ ]:
import json

import pandas as pd

from ruya.config import ayarlar

ayar = ayarlar()
DENEME = ayar.veri("deneyler", "01")
DENEME.mkdir(parents=True, exist_ok=True)

NABLUSI_SATIRLAR = (ayar.ham_klasoru / "nablusi.txt").read_text(encoding="utf-8").splitlines()
paragraflar = nablusi_paragraflari(NABLUSI_SATIRLAR)

print(f"Toplam paragraf: {len(paragraflar)}")
print("Tür sayıları:")
print(pd.Series([p["tur"] for p in paragraflar]).value_counts().to_string())
bab_sayilari = pd.Series([p["bab"] for p in paragraflar]).value_counts()
print(f"Bab sayısı: {len(bab_sayilari)}")
print("Bab başına paragraf:")
print(bab_sayilari.to_string())

In [ ]:
BAB_ANAHTARI = "الحاء"  # Bab adı farklıysa burayı değiştir

bab_adlari = list(dict.fromkeys(p["bab"] for p in paragraflar if p["bab"]))
bab = next((ad for ad in bab_adlari if BAB_ANAHTARI in ad), None)
if bab is None:
    raise ValueError(f"'{BAB_ANAHTARI}' içeren bab yok. Bablar: {bab_adlari}")

ornek = [p for p in paragraflar if p["bab"] == bab][:50]
(DENEME / "h1_ornek.json").write_text(
    json.dumps(ornek, ensure_ascii=False, indent=1), encoding="utf-8"
)

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 60)
tablo = pd.DataFrame(
    [
        {
            "no": p["no"],
            "tur": p["tur"],
            "baslik": p["baslik"],
            "sayfa": p["sayfa"],
            "satir_sayisi": p["satir_sayisi"],
            "metin": p["metin"][:150],
        }
        for p in ornek
    ]
)
print(f"Bab: {bab} · {len(ornek)} paragraf")
tablo

In [ ]:
def ham_goster(no: int) -> None:
    """Paragrafın ham satırlarını elle denetim için yazdırır."""
    p = paragraflar[no]
    bas = p["satir"] - 1
    print(f"no={no} · bab={p['bab']} · tur={p['tur']} · sayfa={p['sayfa']}")
    for i in range(bas, bas + p["satir_sayisi"]):
        print(f"{i + 1:5}  {NABLUSI_SATIRLAR[i]}")

In [ ]:
# ELLE DOLDUR: sınırı ya da türü yanlış paragrafların numaraları
HATALI_H1: list[int] = []

dogru = len(ornek) - len(HATALI_H1)
print(f"H1: {dogru}/{len(ornek)} doğru (eşik ≥ 48)")

## 3. Miller

Beklenen: ~2256 madde. Örnek, 200–1500 karakterlik maddelerden sabit tohumla seçilir.

In [ ]:
import random

miller_metni = (ayar.ham_klasoru / "miller.txt").read_text(encoding="utf-8")
miller = miller_maddeleri(miller_metni)
print(f"Miller madde sayısı: {len(miller)}")

uygun = [m for m in miller if 200 <= len(m["metin"]) <= 1500]
miller_ornek = random.Random(926).sample(uygun, 10)
for m in miller_ornek:
    print()
    print(f"{m['baslik']} (satır {m['satir']})")
    print(m["metin"][:200])

## 4. LLM çağrısı ve önbellek

İstekler `LLMIstemcisi.toplu` ile eşzamanlı gönderilir; yanıtlar istemcinin disk önbelleğine (data/cache/llm) yazılır, notebook yeniden çalıştırıldığında geçide tekrar gidilmez. Bu bölümde rüya metni gönderilmez.

In [ ]:
import json
import time

from IPython.display import HTML, display

from ruya.config import MODEL_CEVIRI, MODEL_SEMBOL_ATAMA, REPO_KOKU
from ruya.hatalar import RuyaHatasi
from ruya.llm.client import LLMIstemcisi

istemci = LLMIstemcisi()
OLCUMLER: list[dict] = []


def istek_kur(model, sistem, kullanici, sema=None, max_tokens=8000) -> dict:
    """OpenAI sohbet gövdesi: sistem ve kullanıcı mesajı, isteğe bağlı JSON şeması."""
    istek = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": [
            {"role": "system", "content": sistem},
            {"role": "user", "content": kullanici},
        ],
    }
    if sema is not None:
        istek["response_format"] = {
            "type": "json_schema",
            "json_schema": {"name": "yanit", "schema": sema, "strict": True},
        }
    return istek


def toplu_cagir(ad: str, istekler: dict[str, dict], eszamanli: int = 5) -> dict:
    """İstekleri eşzamanlı gönderir; hataları ayıklar ve ölçümü OLCUMLER'e ekler."""
    basla = time.perf_counter()
    sonuclar = istemci.toplu(istekler, eszamanli=eszamanli)
    sure_sn = time.perf_counter() - basla

    yanitlar = {}
    for anahtar, sonuc in sonuclar.items():
        if isinstance(sonuc, RuyaHatasi):
            print(f"{ad}/{anahtar}: {sonuc}")
        else:
            yanitlar[anahtar] = sonuc

    gonderilen = sum(1 for y in yanitlar.values() if not y.kullanim.disk_onbelleginden)
    olcum = {
        "ad": ad,
        "istek": len(istekler),
        "gonderilen": gonderilen,
        "sure_sn": sure_sn,
        "girdi": sum(
            y.kullanim.girdi_token + y.kullanim.onbellek_okuma_token for y in yanitlar.values()
        ),
        "cikti": sum(y.kullanim.cikti_token for y in yanitlar.values()),
    }
    OLCUMLER.append(olcum)
    print(
        f"{ad}: {len(yanitlar)}/{len(istekler)} yanıt · {gonderilen} geçide gitti · "
        f"{sure_sn:.1f} sn · girdi {olcum['girdi']:,} · çıktı {olcum['cikti']:,} token"
    )
    return yanitlar

## 5. H2 · Sembol atama (10 alt paragraf)

"ح" babındaki 50 paragraftan 10 alt paragraf seçilir; model her birinin hangi sembolü yorumladığına karar verir. Doğru cevap elle işaretlenir (eşik ≥ 8/10).

In [ ]:
alt_ornek = [p for p in ornek if p["tur"] == "alt"][:10]


def onceki_baslik(p: dict) -> str | None:
    """Bu paragraftan önceki en yakın başlıklı paragrafın başlığı (no'ya göre geriye)."""
    for aday in reversed(paragraflar[: p["no"]]):
        if aday["tur"] == "baslikli":
            return aday["baslik"]
    return None


SISTEM_ATAMA = """Aşağıdaki paragraf Nablusi'nin Ta'tîrü'l-enâm adlı eserinden alınmıştır.
Eser, Arapça rüya sembollerini alfabetik sırayla işleyen bir tabir sözlüğüdür.

"ومن رأى" ile başlayan alt paragraflar çoğunlukla kendinden önceki başlıklı maddeyi
sürdürür; ancak bazen yeni bir sembol başlatır. Paragrafın hangi sembolü yorumladığına
karar ver.

Yanıtı yalnızca istenen JSON şemasıyla ver."""

SEMA_ATAMA = {
    "type": "object",
    "additionalProperties": False,
    "required": ["sembol_ar", "sembol_tr", "onceki_basliga_ait", "gerekce", "ozet_tr"],
    "properties": {
        "sembol_ar": {"type": "string"},
        "sembol_tr": {"type": "string"},
        "onceki_basliga_ait": {"type": "boolean"},
        "gerekce": {"type": "string"},
        "ozet_tr": {"type": "string"},
    },
}

istekler = {
    str(p["no"]): istek_kur(
        MODEL_SEMBOL_ATAMA,
        SISTEM_ATAMA,
        f"Önceki başlık: {onceki_baslik(p) or '-'}\n\nParagraf: {p['metin']}",
        sema=SEMA_ATAMA,
        max_tokens=4000,
    )
    for p in alt_ornek
}
yanitlar = toplu_cagir("h2", istekler)

atama_sonuclari = []
for p in alt_ornek:
    yanit = yanitlar.get(str(p["no"]))
    if yanit is None:
        continue
    atama_sonuclari.append({"no": p["no"], "onceki": onceki_baslik(p), **yanit.json()})

pd.DataFrame(atama_sonuclari)[
    ["no", "onceki", "sembol_ar", "sembol_tr", "onceki_basliga_ait", "ozet_tr", "gerekce"]
]

In [ ]:
# ELLE DOLDUR: her alt paragrafın ataması doğru mu? (True / False)
DOGRU_H2: list[bool | None] = [None] * 10

if None in DOGRU_H2:
    print(f"H2: {DOGRU_H2.count(None)} hücre hâlâ boş; hepsini doldur.")
else:
    print(f"H2: {sum(DOGRU_H2)}/10 doğru (eşik ≥ 8)")

## 6. H3 · Çeviri (10 Nablusi + 10 Miller)

Puanlama ölçeği: 1 = anlam bozuk, 3 = anlam doğru ama terimler tutarsız, 5 = anlam ve terimler doğru, akıcı. Eşik: ortalama ≥ 3.

In [ ]:
import html

TERIMLER = (REPO_KOKU / "docs" / "terimler.md").read_text(encoding="utf-8").split("\n## 3.")[0]

SISTEM_CEVIRI = f"""Verilen rüya tabiri maddesini Türkçeye çevir.

Kurallar:
- Anlamı koru, hiçbir şey ekleme; süsleme ve yorum katma.
- Kur'an ayeti ve hadis alıntılarını «» içine al.
- Aşağıdaki terim sözlüğündeki karşılıkları tutarlı biçimde kullan.
- Yorum kalıplarını kehanet değil yorum olarak ver:
  "يدل على" → "... ile yorumlanır" / "...ya işaret eder".
- Yalnızca çeviriyi yaz; açıklama ekleme.

Terim sözlüğü:
{TERIMLER}"""

nablusi_ceviri = [p for p in ornek if p["tur"] == "baslikli"][:10]

istekler = {}
for p in nablusi_ceviri:
    istekler[f"n{p['no']}"] = istek_kur(
        MODEL_CEVIRI, SISTEM_CEVIRI, f"Kaynak dil: Arapça\n\n{p['metin']}"
    )
for m in miller_ornek:
    istekler[f"m{m['satir']}"] = istek_kur(
        MODEL_CEVIRI, SISTEM_CEVIRI, f"Kaynak dil: İngilizce\n\n{m['metin']}"
    )
yanitlar = toplu_cagir("h3", istekler)

ceviriler = []
for p in nablusi_ceviri:
    yanit = yanitlar.get(f"n{p['no']}")
    if yanit is None:
        continue
    ceviriler.append(
        {
            "kaynak": "nablusi",
            "no": p["no"],
            "baslik": p["baslik"],
            "asil": p["metin"],
            "ceviri": yanit.icerik,
            "girdi": yanit.kullanim.girdi_token + yanit.kullanim.onbellek_okuma_token,
            "cikti": yanit.kullanim.cikti_token,
        }
    )
for m in miller_ornek:
    yanit = yanitlar.get(f"m{m['satir']}")
    if yanit is None:
        continue
    ceviriler.append(
        {
            "kaynak": "miller",
            "no": m["satir"],
            "baslik": m["baslik"],
            "asil": m["metin"],
            "ceviri": yanit.icerik,
            "girdi": yanit.kullanim.girdi_token + yanit.kullanim.onbellek_okuma_token,
            "cikti": yanit.kullanim.cikti_token,
        }
    )

(DENEME / "ceviriler.json").write_text(
    json.dumps(ceviriler, ensure_ascii=False, indent=1), encoding="utf-8"
)

satirlar = []
for c in ceviriler:
    yon = "rtl" if c["kaynak"] == "nablusi" else "ltr"
    satirlar.append(
        f"<tr><td dir='{yon}'>{html.escape(c['asil'])}</td><td>{html.escape(c['ceviri'])}</td></tr>"
    )
display(HTML("<table>" + "".join(satirlar) + "</table>"))

In [ ]:
# ELLE DOLDUR: her çevirinin puanı (1–5)
PUAN_NABLUSI: list[int | None] = [None] * 10
PUAN_MILLER: list[int | None] = [None] * 10

for ad, puanlar in (("Nablusi", PUAN_NABLUSI), ("Miller", PUAN_MILLER)):
    dolu = [p for p in puanlar if p is not None]
    if dolu:
        print(f"{ad}: ortalama {sum(dolu) / len(dolu):.2f} ({len(dolu)}/10 dolu, eşik ≥ 3)")
    else:
        print(f"{ad}: henüz puan yok (eşik ≥ 3)")

## 7. H4 · Bütün Nablusi'nin çevirisi: token ve süre

Ücretsiz dönemde asıl sınırlar günlük kota (15 milyon token) ve süredir. Tahmin H3'teki 10 Nablusi çağrısından ölçülen karakter başına token ve istek başına etkin süreyle yapılır. Eşik: toplam ≤ 30 milyon token (iki günlük kota) ve 5 eşzamanlı istekle ≤ 24 saat. Not: süre ölçümü yalnızca yanıtlar önbellekten gelmediyse anlamlıdır; önbellekten geldiyse önceki ölçüm kullanılır.

In [ ]:
nablusi_c = [c for c in ceviriler if c["kaynak"] == "nablusi"]
karakter = sum(len(c["asil"]) for c in nablusi_c)
GIRDI_KARAKTER = sum(c["girdi"] for c in nablusi_c) / karakter
CIKTI_KARAKTER = sum(c["cikti"] for c in nablusi_c) / karakter
TOPLAM_KARAKTER = sum(len(p["metin"]) for p in paragraflar)
N = len(paragraflar)
H4_TOKEN = (GIRDI_KARAKTER + CIKTI_KARAKTER) * TOPLAM_KARAKTER

# İstek başına etkin süre: H3 ölçümünden (eşzamanlılık 5 zaten içinde).
h3_olcum = next(
    (o for o in reversed(OLCUMLER) if o["ad"] == "h3" and o["gonderilen"] > 0), None
)
sure_dosyasi = DENEME / "h4_sure.json"
if h3_olcum is not None:
    etkin = h3_olcum["sure_sn"] / h3_olcum["gonderilen"]
    sure_dosyasi.write_text(json.dumps({"etkin_sn": etkin}, indent=1), encoding="utf-8")
elif sure_dosyasi.exists():
    etkin = json.loads(sure_dosyasi.read_text(encoding="utf-8"))["etkin_sn"]
else:
    etkin = None

H4_SAAT = N * etkin / 3600 if etkin else None

print(f"Paragraf sayısı N: {N:,}")
print(f"Toplam karakter: {TOPLAM_KARAKTER:,}")
print(f"Girdi tokenı / karakter: {GIRDI_KARAKTER:.3f}")
print(f"Çıktı tokenı / karakter: {CIKTI_KARAKTER:.3f}")
print(f"H4 token tahmini: {H4_TOKEN / 1e6:.1f} M")
if H4_SAAT is None:
    print("H4 süre tahmini: ? (H3 ölçümü yok, önbellekteki süre de yok)")
else:
    print(f"H4 süre tahmini: {H4_SAAT:.1f} saat (istek başına {etkin:.1f} sn)")
print("H4 eşiği: ≤ 30 M token ve ≤ 24 saat")

print(
    f"Bu notebook'ta harcanan: {sum(o['girdi'] for o in OLCUMLER):,} girdi + "
    f"{sum(o['cikti'] for o in OLCUMLER):,} çıktı token"
)

## 8. Sonuç

Aşağıdaki tablo `docs/deneyler/01-veri-ve-ceviri.md` dosyasının "Sonuç" bölümüne yapıştırılır.

In [ ]:
def _ortalama(puanlar: list[int | None]) -> float | None:
    dolu = [p for p in puanlar if p is not None]
    return sum(dolu) / len(dolu) if dolu else None


h1_dogru = len(ornek) - len(HATALI_H1)
h1 = f"{h1_dogru}/{len(ornek)}"
h1_tuttu = "evet" if h1_dogru >= 48 else "hayır"

h2 = "?" if None in DOGRU_H2 else f"{sum(DOGRU_H2)}/10"
h2_tuttu = "?" if None in DOGRU_H2 else ("evet" if sum(DOGRU_H2) >= 8 else "hayır")

h3_nablusi = _ortalama(PUAN_NABLUSI)
h3_miller = _ortalama(PUAN_MILLER)
if h3_nablusi is None or h3_miller is None:
    h3, h3_tuttu = "?", "?"
else:
    h3 = f"Nablusi {h3_nablusi:.2f} · Miller {h3_miller:.2f}"
    h3_tuttu = "evet" if min(h3_nablusi, h3_miller) >= 3 else "hayır"

# H4 tahmini 7. bölümde hesaplanır; o hücre çalışmadıysa "?" yazılır.
h4_token = globals().get("H4_TOKEN")
h4_saat = globals().get("H4_SAAT")
if h4_token is None or h4_saat is None:
    h4, h4_tuttu = "?", "?"
else:
    h4 = f"{h4_token / 1e6:.1f} M token · {h4_saat:.1f} saat"
    h4_tuttu = "evet" if h4_token <= 30e6 and h4_saat <= 24 else "hayır"

print("| Hipotez | Ölçülen | Tuttu mu? |")
print("|---|---|---|")
print(f"| H1 | {h1} doğru | {h1_tuttu} |")
print(f"| H2 | {h2} doğru | {h2_tuttu} |")
print(f"| H3 | {h3} | {h3_tuttu} |")
print(f"| H4 | {h4} | {h4_tuttu} |")